# Mock VOAI: từ đề lạ đến submission tái lập

**Thời lượng:** Chia 6 phiên × 60 phút  
**Chế độ:** SOLO-90 — không dùng AI sinh code hoặc pseudocode.

Notebook này là bài thực hành có chủ đích, không phải lời giải mẫu.


## Mục tiêu

- Đọc metric và dựng baseline trong 30 phút
- Thiết kế split chống leakage
- Lưu prediction đúng format và tái lập bằng seed


## Trực giác cốt lõi

Trong đề thi, baseline sớm cho vòng lặp nhanh. Thứ tự an toàn: hiểu metric → kiểm tra dữ liệu → split → baseline → error analysis → một thay đổi mỗi lần → xác nhận validation → đóng gói submission.

Trước khi chạy cell tiếp theo, hãy viết một dự đoán vào sổ học.


In [ ]:
import numpy as np
from sklearn.datasets import make_classification
X,y=make_classification(n_samples=900,n_features=24,n_informative=8,weights=[.72,.28],flip_y=.04,random_state=42)
print(X.shape, np.bincount(y))


## Tự cài đặt

Không mở đáp án. Đầu tiên hãy ghi input/output, shape và edge cases; sau đó mới viết code.


In [ ]:
# PHIEN 1: data audit (phan bo lop, gia tri thieu, trung lap) va chon metric
# PHIEN 2: split co stratify thanh train/valid/test; KHONG duoc chong chi so
# PHIEN 3: baseline don gian, phai vuot muc doan lop da so
# PHIEN 4: error analysis va dung MOT cai tien
# PHIEN 5: ablation + xac nhan tai lap bang seed
# PHIEN 6: xuat submission.csv va bao cao 1 trang

SEED = 42          # TODO: dung SEED nay o moi cho co ngau nhien
train_idx = None   # TODO: mang chi so
valid_idx = None   # TODO: mang chi so, khong giao voi train
test_idx = None    # TODO: mang chi so, giu kin toi phien cuoi
model = None       # TODO: mot estimator da fit, co .fit/.predict
validation_score = None  # TODO: F1 tren valid_idx, tu tinh lai duoc


## Visible tests

Các test dưới đây chỉ kiểm tra interface và trường hợp cơ bản. Notebook không có hidden test phía máy chủ; sau khi test đạt, hãy tự viết thêm edge case và lưu evidence vào bản sao của bạn.


In [ ]:
# Audit checkpoint: mot placeholder rong (khong co fit/predict) KHONG duoc di qua.
assert model is not None, 'Can mot mo hinh da fit'
assert hasattr(model, 'fit') and hasattr(model, 'predict'), 'model phai co interface fit/predict'
assert callable(model.predict), 'model.predict phai goi duoc'

for _name in ('train_idx', 'valid_idx', 'test_idx', 'SEED'):
    assert globals().get(_name) is not None, 'thieu ' + _name

train_idx = np.asarray(train_idx); valid_idx = np.asarray(valid_idx); test_idx = np.asarray(test_idx)
assert len(set(train_idx) & set(valid_idx)) == 0, 'train va validation trung chi so - leakage'
assert len(set(train_idx) & set(test_idx)) == 0, 'train va mock-test trung chi so - leakage'
assert len(set(valid_idx) & set(test_idx)) == 0, 'validation va mock-test trung chi so - leakage'
assert len(train_idx) + len(valid_idx) + len(test_idx) == len(y), 'split khong phu het du lieu'

pred_valid = np.asarray(model.predict(X[valid_idx]))
assert pred_valid.shape[0] == len(valid_idx), 'predict tra sai so luong'

from sklearn.metrics import f1_score
measured = f1_score(y[valid_idx], pred_valid)
assert 0 <= validation_score <= 1
assert abs(measured - validation_score) < 1e-6, 'validation_score khong khop metric tinh lai'

majority = np.full(len(valid_idx), np.bincount(y[train_idx]).argmax())
assert measured > f1_score(y[valid_idx], majority), 'chua vuot baseline doan lop da so'
assert np.array_equal(pred_valid, np.asarray(model.predict(X[valid_idx]))), 'predict khong tat dinh'

import os
assert os.path.exists('submission.csv'), 'thieu artifact submission.csv'
print('OK Audit dat - F1 validation', round(measured, 4), 'seed', SEED)


## Deep 60

Lặp mock với ảnh, văn bản hoặc audio từ kho IOAI chính thức. Không reuse test để tuning. Nhật ký phải ghi thử nghiệm, seed, metric và quyết định giữ/bỏ.


## Exit ticket

1. Tôi có thể giải thích thuật toán mà không nhìn code không?  
2. Edge case nào làm bản đầu tiên sai?  
3. Độ phức tạp thời gian/bộ nhớ là gì?  
4. Tôi sẽ ôn lại điều gì vào ngày +1, +7 và +21?
